# 08 — Download Mel+Nevus Histo Dataset

**Issue:** D4.1 (#137)  
**Dataset:** Melanoma and Nevus Dermoscopy Images with Confirmed Histopathological Diagnosis  
**ISIC Collection ID:** 294  
**Expected:** 18,133 images (7,191 melanoma, 39.7%)  
**Destination:** `data/mel_nevus_histo/`

In [2]:
import pandas as pd
from pathlib import Path

ROOT = Path("/Users/rehmaaziz/revela")
METADATA_PATH = ROOT / "data/mel_nevus_histo/metadata.csv"
IMAGES_DIR    = ROOT / "data/mel_nevus_histo/images"

assert METADATA_PATH.exists(), f"Metadata not found at {METADATA_PATH}"
assert IMAGES_DIR.exists(), f"Images dir not found at {IMAGES_DIR}"

## 1. Metadata — row count assert

In [3]:
df = pd.read_csv(METADATA_PATH, low_memory=False)
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

assert len(df) == 18_133, f"Expected 18,133 rows, got {len(df)}"
print("\nRow count OK — 18,133")

Rows: 18,133
Columns: ['isic_id', 'attribution', 'copyright_license', 'acquisition_day', 'age_approx', 'anatom_site_1', 'anatom_site_2', 'anatom_site_3', 'anatom_site_4', 'anatom_site_5', 'anatom_site_special', 'clin_size_long_diam_mm', 'concomitant_biopsy', 'dermoscopic_type', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_4', 'diagnosis_5', 'diagnosis_confirm_type', 'family_hx_mm', 'fitzpatrick_skin_type', 'image_manipulation', 'image_type', 'lesion_id', 'mel_mitotic_index', 'mel_thick_mm', 'mel_ulcer', 'melanocytic', 'patient_id', 'personal_hx_mm', 'sex']

Row count OK — 18,133


## 2. Class distribution — diagnosis_2 / diagnosis_3

In [4]:
print("=== diagnosis_2 ===")
print(df["diagnosis_2"].value_counts(dropna=False).to_string())
print()
print("=== diagnosis_3 ===")
print(df["diagnosis_3"].value_counts(dropna=False).to_string())

=== diagnosis_2 ===
diagnosis_2
Benign melanocytic proliferations                       10826
Malignant melanocytic proliferations (Melanoma)          7191
Indeterminate melanocytic proliferations                   80
Collision - Only benign proliferations                     15
Collision - At least one malignant proliferation           11
Benign epidermal proliferations                             6
Flat melanotic pigmentations - not melanocytic nevus        4

=== diagnosis_3 ===
diagnosis_3
Nevus                                                             10826
Melanoma, NOS                                                      5359
Melanoma in situ                                                   1035
Melanoma Invasive                                                   797
Atypical melanocytic neoplasm                                        70
NaN                                                                  26
Epidermal nevus                                                      

In [5]:
# Melanoma count from diagnosis_3
mel_mask = df["diagnosis_3"].str.lower().str.contains("melanoma", na=False)
mel_count = mel_mask.sum()
mel_pct = mel_count / len(df) * 100
print(f"Melanoma rows: {mel_count:,} ({mel_pct:.1f}%)")
print(f"Expected:      7,191 (39.7%)")

Melanoma rows: 7,191 (39.7%)
Expected:      7,191 (39.7%)


## 3. Image count and zero-byte check

In [6]:
# Use rglob since IMAGES_DIR is a symlink and we want to traverse into it
images = [p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg"}]
print(f"Images on disk: {len(images):,}")
assert len(images) == 18_133, f"Expected 18,133 images, got {len(images)}"
print("Image count OK — 18,133")

Images on disk: 18,133
Image count OK — 18,133


In [7]:
zero_byte = [p for p in images if p.stat().st_size == 0]
print(f"Zero-byte files: {len(zero_byte)}")
assert len(zero_byte) == 0, f"Found {len(zero_byte)} zero-byte files: {zero_byte[:5]}"
print("Zero-byte check OK")

Zero-byte files: 0
Zero-byte check OK


## 4. Summary table

In [8]:
bcn_overlap_estimate = 5_633
post_filter_estimate = len(df) - bcn_overlap_estimate

summary = pd.DataFrame({
    "Stage": ["Raw MNH", "After BCN20000 dedup (est.)"],
    "Total images": [f"{len(df):,}", f"~{post_filter_estimate:,}"],
    "Melanoma": [f"{mel_count:,}", "~4,334"],
    "Melanoma %": [f"{mel_pct:.1f}%", "~34.7%"],
})
print(summary.to_string(index=False))

                      Stage Total images Melanoma Melanoma %
                    Raw MNH       18,133    7,191      39.7%
After BCN20000 dedup (est.)      ~12,500   ~4,334     ~34.7%
